In [31]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    return -par[2]**2 / ((p - par[0]) - par[2])**2

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 5]
print(p)
q = df[:, 6]


err_p = df[:, 8]
err_q =  df[:, 7]


p = np.delete(p, [0, 62, 88] )
q = np.delete(q, [0, 62, 88]  )
err_p = np.delete(err_p, [0, 62, 88] )
err_q = np.delete(err_q, [0, 62, 88] )


#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

[ 232.5  233.5  234.5  235.5  236.5  237.5  238.5  239.5  240.5  241.5
  242.5  243.5  244.5  245.5  246.5  247.5  248.5  249.5  250.5  251.5
  252.5  253.5  254.5  255.5  256.5  257.5  258.5  259.5  260.5  261.5
  262.5  263.5  264.5  265.5  267.5  269.5  271.5  273.5  275.5  277.5
  280.5  281.5  282.5  283.5  284.5  285.5  286.5  287.5  288.5  289.5
  290.5  291.5  292.5  293.5  294.5  295.5  297.5  299.5  301.5  306.5
  311.5  316.5  321.5  326.5  331.5  336.5  341.5  346.5  351.5  356.5
  361.5  366.5  371.5  376.5  381.5  386.5  391.5  396.5  401.5  406.5
  411.5  416.5  421.5  426.5  431.5  441.5  451.5  461.5  471.5  481.5
  491.5  501.5  511.5  521.5  531.5  541.5  551.5  561.5  571.5  581.5
  601.5  621.5  641.5  661.5  681.5  731.5  781.5  831.5  881.5  981.5
 1081.5 1181.5 1281.5 1381.5 1481.5 1631.5]
Fit status: 0
Parametri stimati:
A = -1.0080348131154342 ± 0.2179267858590797
B = 1.9155972596739395 ± 0.15616923546674824
C = 200.7759102272441 ± 0.09546267995490164
Chi2 = 1

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


## Fit separati a p = 400

Come consigliato dal prof: stesso modello e stessi dati, ma due fit distinti (p < 400 e p > 400), per vedere se A viene uguale e se il chi² migliora.

Le variabili nuove hanno suffisso `_1` / `_2` (e `gr_split`, `c_split`), quindi le celle successive (compatibilità con 0, residui, istogramma) continuano a lavorare sul fit completo di prima.

In [ ]:
# ---- Fit separati: p < p_cut e p > p_cut ----
# Stessi dati e stesso modello del fit completo: cambia solo l'intervallo di p.
# Con l'opzione "R" ROOT usa solo i punti dentro il range del TF1, quindi basta dare a ogni TF1 il suo range.
p_cut = 400

# valori iniziali = parametri del fit completo (partendo da 0,0,0 il fit su p<400 puo' finire in un minimo sbagliato)
model_fit_1 = ROOT.TF1('model_1', model, min(p), p_cut, 3)      # parte con p < 400
model_fit_1.SetParameters(par[0], par[1], par[2])
model_fit_1.SetParNames("A", "B", "C")
model_fit_1.SetLineColor(ROOT.kRed + 1)
model_fit_1.SetLineWidth(2)

model_fit_2 = ROOT.TF1('model_2', model, p_cut, max(p), 3)      # parte con p > 400
model_fit_2.SetParameters(par[0], par[1], par[2])
model_fit_2.SetParNames("A", "B", "C")
model_fit_2.SetLineColor(ROOT.kGreen + 2)
model_fit_2.SetLineWidth(2)

gr_split = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr_split.SetTitle("Fit separati; p; q")
gr_split.SetMarkerStyle(20)
gr_split.SetMarkerSize(0.7)
gr_split.SetMarkerColor(ROOT.kAzure + 2)
gr_split.SetLineColor(ROOT.kAzure + 3)
gr_split.SetLineWidth(1)

# "N" = non salva/disegna la funzione dentro il grafico (le disegno io nella cella dopo)
fit_res_1 = gr_split.Fit(model_fit_1, "SRN")
fit_res_2 = gr_split.Fit(model_fit_2, "SRN")

par_1 = model_fit_1.GetParameters()
par_2 = model_fit_2.GetParameters()
errors_1 = model_fit_1.GetParErrors()
errors_2 = model_fit_2.GetParErrors()
chi2_1, ndf_1 = fit_res_1.Chi2(), fit_res_1.Ndf()
chi2_2, ndf_2 = fit_res_2.Chi2(), fit_res_2.Ndf()

print(f"Fit status: {fit_res_1.Status()}, {fit_res_2.Status()}")
print()
print(f"Fit 1 (p < {p_cut}):")
print(f"A = {par_1[0]} ± {errors_1[0]}")
print(f"B = {par_1[1]} ± {errors_1[1]}")
print(f"C = {par_1[2]} ± {errors_1[2]}")
print(f"Chi2 = {chi2_1}   ndf = {ndf_1}")
print()
print(f"Fit 2 (p > {p_cut}):")
print(f"A = {par_2[0]} ± {errors_2[0]}")
print(f"B = {par_2[1]} ± {errors_2[1]}")
print(f"C = {par_2[2]} ± {errors_2[2]}")
print(f"Chi2 = {chi2_2}   ndf = {ndf_2}")

# ---- I parametri vengono uguali nei due fit? (differenza in unita' di sigma, soglia 3 sigma come sopra) ----
print()
for i, nome in enumerate(["A", "B", "C"]):
    diff = np.abs(par_1[i] - par_2[i])
    sigma = np.sqrt(errors_1[i]**2 + errors_2[i]**2)
    if diff < 3*sigma:
        print(f"{nome}: differenza = {diff:.3f} ({diff/sigma:.1f} sigma) -> compatibili")
    else:
        print(f"{nome}: differenza = {diff:.3f} ({diff/sigma:.1f} sigma) -> NON compatibili")

# ---- Il chi quadro migliora? (confronto col fit unico fatto sopra) ----
chi2_tot = model_fit.GetChisquare()      # fit unico
ndf_tot = model_fit.GetNDF()
chi2_split = chi2_1 + chi2_2             # i due fit insieme
ndf_split = ndf_1 + ndf_2
print()
print(f"Fit unico    : chi2 = {chi2_tot:.1f}  ndf = {ndf_tot}  chi2/ndf = {chi2_tot/ndf_tot:.1f}")
print(f"Due fit somma: chi2 = {chi2_split:.1f}  ndf = {ndf_split}  chi2/ndf = {chi2_split/ndf_split:.1f}")

In [ ]:
# ---- Grafico: dati + i due fit ----
c_split = ROOT.TCanvas("c_split", "Fit separati", 800, 600)
c_split.SetGrid()

gr_split.Draw("AP")
model_fit_1.Draw("same")
model_fit_2.Draw("same")

leg_split = ROOT.TLegend(0.40, 0.72, 0.89, 0.89)
leg_split.AddEntry(gr_split, "Dati", "pe")
leg_split.AddEntry(model_fit_1, f"Fit p < {p_cut}: A = {par_1[0]:.1f} #pm {errors_1[0]:.1f}, #chi^{{2}}/ndf = {chi2_1:.0f}/{ndf_1}", "l")
leg_split.AddEntry(model_fit_2, f"Fit p > {p_cut}: A = {par_2[0]:.1f} #pm {errors_2[0]:.1f}, #chi^{{2}}/ndf = {chi2_2:.0f}/{ndf_2}", "l")
leg_split.Draw()

c_split.Draw()
c_split.SaveAs("fit_root_split.png")

In [32]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if np.abs(par[0]) < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A è compatibile con 0
---
B è compatibile con 0


In [33]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff



#residuals_norm = np.delete(residuals_norm, [0,1, 10, 12, 14, 43, 44, 52])

#print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")


print()
print()

for i in np.arange(0,len(residuals_norm)):
    if residuals_norm[i] < -0.8:
        print (residuals_norm[i])
        print(i)

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).


Fit status: 0
Parametri stimati:
const = 63.96419755302867 ± 8.027487474804495
#mu = 4.671411911988578 ± 0.2483863793837035
#sigma = 2.0008682150566566 ± 0.13543551143679167
Chi2 = 32.95106886745551


-1.560947139141609
66
-8.50884657347015
67
-4.415992710736622
68
-7.2773867077429415
70
-14.758053391815066
71
-24.7354673563883
72
-31.54283024534322
73
-52.15641271616138
74
-63.54373863797072
75
-15.478694338513428
76
-9.824296718854493
91
-2.4343207454161866
96
-10.775443930385936
98
-4.305405917176946
102
-8.481044829162347
103
-4.323093667920824
104
-10.868568407049175
106
-12.189698095197285
107
-6.5117717607665995
108
-6.9536297966602705
110
-1.228437271012943
111
-7.332867833165924
112
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      32.9511
NDf                       =           10
Edm                       =  6.86552e-07
NCalls                    =          160
const                     =      63.9642   +/-   8.02749     
mu

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file hist.png has been created


In [34]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu NON è compatibile con 0
---
sigma NON è compatibile con 1


In [35]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("APX")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

[ 1.24813638e+02  1.51307918e+02  1.17471246e+02  1.16414333e+02
  1.11553803e+02  1.08673567e+02  9.88823325e+01  9.98754904e+01
  8.79555662e+01  8.34044027e+01  9.20247699e+01  7.40481330e+01
  7.59082794e+01  6.77620125e+01  7.88058643e+01  7.67864941e+01
  7.28328582e+01  6.68283590e+01  6.58273810e+01  5.96389550e+01
  6.18450730e+01  5.79047312e+01  6.02317711e+01  5.40373896e+01
  5.08328090e+01  4.68069881e+01  4.37536938e+01  4.69149196e+01
  4.66328855e+01  4.25812539e+01  4.61854570e+01  4.19552729e+01
  4.03632920e+01  3.80684632e+01  3.66430611e+01  3.47331151e+01
  3.44956947e+01  3.24119575e+01  3.31574183e+01  2.78667903e+01
  2.94676544e+01  2.79362793e+01  2.23361119e+01  1.99966339e+01
  2.45458486e+01  2.14827956e+01  2.23023257e+01  2.39680116e+01
  2.15350318e+01  2.14861241e+01  2.33201333e+01  1.90946890e+01
  2.12097943e+01  1.97639688e+01  2.12088937e+01  1.98110867e+01
  1.95937799e+01  1.60281566e+01  1.55984131e+01  1.50151327e+01
  9.79678610e-01  2.22839

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created
